In [1]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Funciones When coalesce y lit").getOrCreate()

In [3]:
data = spark.read.parquet("./data/")

## Funciones `when` y `otherwise`

Cuando necesitamos crear lógica condicional en Spark (el equivalente a un bloque `if-elif-else` en Python o un `CASE WHEN` en SQL), utilizamos las funciones integradas **`when`** y **`otherwise`**.

Estas funciones nos permiten evaluar una o varias condiciones en orden y asignar valores específicos a una nueva columna según el resultado de dicha evaluación.

---

##  Reglas Importantes de Uso

1. **Estructura Encadenada:** La función `when` se puede encadenar tantas veces como sea necesario para evaluar múltiples condiciones lógicas (funcionando como un `elif`).
2. **El Cierre con `otherwise`:** El método `otherwise` sirve para definir el valor por defecto si ninguna de las condiciones anteriores se cumple (funcionando como el `else`).
3. **Manejo de Nulos:** Si no se define un método `otherwise` al final de la cadena y ninguna condición se cumple, Spark rellenará automáticamente esa fila con un valor nulo (`Null`).

---

## Ejemplo Práctico en PySpark

Imagina que estás analizando el DataFrame de los jugadores y quieres crear una nueva columna categórica llamada `tipo_tarjeta` basada en la cantidad de tarjetas rojas que ha recibido cada jugador:

```python
from pyspark.sql.functions import when, col, lit

# Aplicamos condiciones secuenciales con when y otherwise
df_con_categorias = df.withColumn(
    "tipo_tarjeta",
    when(col("cantidad_tarjetas_rojas") == 0, lit("Limpio")) \
    .when(col("cantidad_tarjetas_rojas") == 1, lit("Advertido")) \
    .when(col("cantidad_tarjetas_rojas") == 2, lit("Peligroso")) \
    .otherwise(lit("Expulsado Frecuente"))
)

# Mostrar el resultado
df_con_categorias.select("nombre_de_pila", "cantidad_tarjetas_rojas", "tipo_tarjeta").show()

In [4]:
data.show()

+------+----+
|nombre|pago|
+------+----+
|  Jose|   1|
| Julia|   2|
| Katia|   1|
|  NULL|   3|
|  Raul|   3|
+------+----+



In [5]:
from pyspark.sql.functions import when, coalesce, lit

In [ ]:
# When funciona con la condicional when y con el valor por defecto de la condicion otherwise

In [7]:
from pyspark.sql.functions import when, coalesce, lit, col

data.select(
    col("nombre"),
    when(col("pago") == 1, "pagado")
        .when(col("pago") == 2, "sin pagar")
        .otherwise("no iniciar")
        .alias("pago")
).show()

+------+----------+
|nombre|      pago|
+------+----------+
|  Jose|    pagado|
| Julia| sin pagar|
| Katia|    pagado|
|  NULL|no iniciar|
|  Raul|no iniciar|
+------+----------+



## Coalesce

Al trabajar con conjuntos de datos, una de las mejores prácticas consiste en gestionar correctamente los valores nulos (`Null`). Una de las formas más comunes de manejarlos es convertirlos en otros valores alternativos que tengan sentido o representen algo específico dentro de nuestra lógica de procesamiento de negocio.

Para resolver esto, Spark proporciona la función **`coalesce`**. Esta función toma una o más columnas como argumentos y evalúa sus filas de izquierda a derecha, **devolviendo el primer valor que no sea nulo** que encuentre en su camino.

---

## # Reglas Importantes de Uso

1. **Tipo de datos de los argumentos:** Cada uno de los argumentos pasados dentro de la función `coalesce` debe ser estrictamente de **tipo columna** (`Column`).
2. **Uso de valores literales (`lit`):** Si deseas rellenar los espacios nulos con un valor estático por defecto (por ejemplo, una cadena de texto como `"Sin Información"` o un número como `0`), no puedes pasar el valor directamente como un texto plano. Es obligatorio envolverlo utilizando la función **`lit()`** para transformarlo en un objeto tipo columna.

---

## Ejemplo Práctico en PySpark

Imagina que tienes un DataFrame donde algunos jugadores no tienen informado su nombre de pila (`first_name`) y quieres rellenar esos vacíos con su apellido (`last_name`). Si el apellido también es nulo, quieres colocar por defecto la palabra `"Anónimo"`.

```python
from pyspark.sql.functions import coalesce, col, lit

# Aplicamos coalesce para evaluar las opciones en orden de prioridad
df_limpio = df.withColumn(
    "nombre_final",
    coalesce(
        col("first_name"),       # Primera opción
        col("last_name"),        # Segunda opción (si la primera es nula)
        lit("Anónimo")           # Valor por defecto (si ambas son nulas)
    )
)

In [9]:
# en el caso de que el nombre venga nulo en una de las columnas se le debe poner el atributo sin nombre

data.select(
    coalesce(col("nombre"), lit("sin nombre")).alias("nombre")
).show()

+----------+
|    nombre|
+----------+
|      Jose|
|     Julia|
|     Katia|
|sin nombre|
|      Raul|
+----------+

